# Flower Classification Notebook: ORB + XGBoost
This notebook implements the complete training and evaluation pipeline using real dataset inputs split into Train, Validation, and Test subsets.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import gc
from sklearn.model_selection import train_test_split
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from xgboost import XGBClassifier

In [ ]:
# Configurations
DATASET_DIR = "flower-training"
CLASSES = ["bellflower", "daisy", "dandelion", "lotus", "rose", "sunflower", "tulip"]
FEATURE_NAME = "ORB"
CLASSIFIER_NAME = "XGBoost"


## 1. Image Preprocessing (3 Steps)
- **Step 1:** Resize image to $256 \times 256$ pixels.
- **Step 2:** Apply Gaussian Blur filter ($5 \times 5$ kernel).
- **Step 3:** Apply HSV-based Green color removal mask to extract the flower region.

In [ ]:
def preprocess_single_image(img, targetedSize=(256, 256)):
    resized_img = cv2.resize(img, targetedSize, interpolation=cv2.INTER_LINEAR)
    blurred_img = cv2.GaussianBlur(resized_img, (5, 5), 0)
    return blurred_img

def extract_flower_mask(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    lower_green = np.array([35, 40, 40])
    upper_green = np.array([85, 255, 255])
    green_mask = cv2.inRange(hsv, lower_green, upper_green)
    flower_mask = cv2.bitwise_not(green_mask)
    return flower_mask

## 2. Feature Extraction
- Keypoint descriptor extractor (SIFT/ORB)
- HSV Color histogram feature extractor

In [ ]:
class DescriptorService:
    def __init__(self):
        self.extractor = cv2.ORB_create(nfeatures=500)
        
    def extract_features(self, image, mask=None):
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image
        _, descriptors = self.extractor.detectAndCompute(gray, mask)
        if descriptors is None:
            return np.zeros((0, 32), dtype=np.float32)
        return descriptors.astype(np.float32)

def extract_hsv_histogram(image, mask=None):
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], mask, [8, 4, 4], [0, 180, 0, 256, 0, 256])
    hist_flat = hist.flatten()
    norm = np.linalg.norm(hist_flat, ord=2)
    if norm > 0:
        hist_flat /= norm
    return hist_flat

def safe_load_image(path):
    try:
        # Use np.fromfile to handle unicode paths safely
        return cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    except Exception:
        return None

## 3. Dataset Loading and Initial Splits
Load real images from the `flower-training` directory, preprocess them, and split into train/test.

In [ ]:
print("Loading dataset...")
image_paths = []
labels = []

for label_idx, cls in enumerate(CLASSES):
    cls_dir = os.path.join(DATASET_DIR, cls)
    if not os.path.exists(cls_dir):
        continue
    for filename in os.listdir(cls_dir):
        if filename.lower().endswith((".jpg", ".jpeg", ".png")):
            image_paths.append(os.path.join(cls_dir, filename))
            labels.append(label_idx)

print(f"Loaded {len(image_paths)} images from {len(CLASSES)} classes.")

# Split: Train (80%) and Test (20%)
X_train_paths, X_test_paths, y_train, y_test = train_test_split(
    image_paths, labels, test_size=0.2, stratify=labels, random_state=42
)

# Further split Train into Train (90% of Train) and Validation (10% of Train)
# This results in 72% Train, 8% Val, 20% Test of overall dataset
X_tr_paths, X_val_paths, y_tr, y_val = train_test_split(
    X_train_paths, y_train, test_size=0.1, stratify=y_train, random_state=42
)

print(f"Train size: {len(X_tr_paths)}, Val size: {len(X_val_paths)}, Test size: {len(X_test_paths)}")

## 4. Feature Extraction & BoVW Vocabulary Fitting
Fit K-Means vocabulary ONLY on the training descriptors, then compute fused BoVW + HSV Histograms for all subsets.

In [ ]:
descriptor_service = DescriptorService()

# Step 1: Sample a subset of training images to fit K-Means visual vocabulary (saves RAM)
print("Sampling training images for vocabulary fitting...")
np.random.seed(42)
fit_paths = np.random.choice(X_tr_paths, min(len(X_tr_paths), 1000), replace=False)

print("Extracting descriptors for K-Means fitting...")
fit_descs = []
for idx, path in enumerate(fit_paths):
    img = safe_load_image(path)
    if img is None:
        continue
    cleaned = preprocess_single_image(img)
    del img
    mask = extract_flower_mask(cleaned)
    descs = descriptor_service.extract_features(cleaned, mask=mask)
    if len(descs) > 0:
        # sample up to 100 descriptors per image
        s_idx = np.random.choice(len(descs), min(len(descs), 100), replace=False)
        fit_descs.append(descs[s_idx])
    del cleaned, mask, descs
    if (idx + 1) % 100 == 0:
        gc.collect()

flat_fit_descs = np.vstack(fit_descs)
print(f"Fitting K-Means vocabulary with 500 clusters on {len(flat_fit_descs)} descriptors...")
kmeans = MiniBatchKMeans(n_clusters=500, random_state=42, batch_size=1000, n_init="auto")
kmeans.fit(flat_fit_descs)
del fit_descs, flat_fit_descs
gc.collect()

# Step 2: Compute fused features by streaming images (uses very little memory)
def get_fused_features(paths):
    features = []
    for idx, path in enumerate(paths):
        img = safe_load_image(path)
        if img is None:
            features.append(np.zeros(628, dtype=np.float32))
            continue
        cleaned = preprocess_single_image(img)
        del img
        mask = extract_flower_mask(cleaned)
        descs = descriptor_service.extract_features(cleaned, mask=mask)
        color = extract_hsv_histogram(cleaned, mask=mask)
        
        # Project descriptors to visual vocabulary
        histogram = np.zeros(500, dtype=np.float32)
        if len(descs) > 0:
            words = kmeans.predict(descs)
            for w in words:
                histogram[w] += 1.0
        norm1 = np.linalg.norm(histogram, ord=1)
        if norm1 > 0:
            histogram /= norm1
            
        hist_l2 = histogram / (np.linalg.norm(histogram, ord=2) + 1e-6)
        color_l2 = color / (np.linalg.norm(color, ord=2) + 1e-6)
        
        fused = np.hstack([hist_l2, 0.5 * color_l2])
        fused /= np.linalg.norm(fused, ord=2) + 1e-6
        features.append(fused)
        
        del cleaned, mask, descs, color, fused
        if (idx + 1) % 100 == 0:
            gc.collect()
            
    return np.array(features)

print("Extracting features for Train split...")
X_train_fused = get_fused_features(X_tr_paths)
print("Extracting features for Validation split...")
X_val_fused = get_fused_features(X_val_paths)
print("Extracting features for Test split...")
X_test_fused = get_fused_features(X_test_paths)

y_train = np.array(y_tr)
y_val = np.array(y_val)
y_test = np.array(y_test)

## 5. Model Training

In [ ]:

# Train XGBoost model with balanced parameters to reduce underfitting
print("Training XGBoost Classifier...")
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=1.0,
    reg_lambda=1.0,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=1
)
model.fit(
    X_train_fused, y_train,
    eval_set=[(X_train_fused, y_train), (X_val_fused, y_val)],
    verbose=False
)


## 6. Model Evaluation
Print classification report and plot confusion matrix.

In [ ]:
preds = model.predict(X_test_fused)
print(f"--- Classification Report: {FEATURE_NAME} + {CLASSIFIER_NAME} ---")
print(classification_report(y_test, preds, target_names=CLASSES))

# Plot Confusion Matrix
cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax, xticks_rotation=45)
plt.title(f"Confusion Matrix: {FEATURE_NAME} + {CLASSIFIER_NAME}")
plt.show()

## 7. Iterative Training: Accuracy and Loss Curves

In [ ]:

# Plot LogLoss Learning Curves from training history
results = model.evals_result()
epochs = len(results['validation_0']['mlogloss'])
x_axis = range(0, epochs)

plt.figure(figsize=(12, 5))

# Plot Log Loss
plt.subplot(1, 2, 1)
plt.plot(results['validation_0']['mlogloss'], label='Train')
plt.plot(results['validation_1']['mlogloss'], label='Val')
plt.legend()
plt.ylabel('Log Loss')
plt.xlabel('Iterations')
plt.title(f'{FEATURE_NAME} + {CLASSIFIER_NAME} Loss Curve')
plt.grid(True)

# Plot accuracy iteration progress
train_accs = []
val_accs = []
for i in range(1, epochs + 1):
    train_preds = model.predict(X_train_fused, iteration_range=(0, i))
    val_preds = model.predict(X_val_fused, iteration_range=(0, i))
    train_accs.append(np.mean(train_preds == y_train))
    val_accs.append(np.mean(val_preds == y_val))

plt.subplot(1, 2, 2)
plt.plot(x_axis, train_accs, label='Train')
plt.plot(x_axis, val_accs, label='Val')
plt.legend()
plt.ylabel('Accuracy')
plt.xlabel('Iterations')
plt.title(f'{FEATURE_NAME} + {CLASSIFIER_NAME} Accuracy Curve')
plt.grid(True)

plt.tight_layout()
plt.show()


## 8. Learning Curves
Shows training and validation accuracy vs. size of dataset subset.

In [ ]:
train_sizes = [0.2, 0.4, 0.6, 0.8, 1.0]
lc_train = []
lc_val = []

for size in train_sizes:
    n_samples = int(size * len(X_train_fused))
    if n_samples < 10:
        n_samples = 10
    X_sub = X_train_fused[:n_samples]
    y_sub = y_train[:n_samples]
    
    if CLASSIFIER_NAME == "RandomForest":
        clf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_leaf=3,
            min_samples_split=6,
            max_features="sqrt",
            class_weight="balanced",
            random_state=42,
            n_jobs=1
        )
    else:
        clf = XGBClassifier(
            n_estimators=100,
            learning_rate=0.05,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=1.0,
            reg_lambda=1.0,
            eval_metric='mlogloss',
            random_state=42,
            n_jobs=1
        )
        
    clf.fit(X_sub, y_sub)
    lc_train.append(clf.score(X_sub, y_sub))
    lc_val.append(clf.score(X_val_fused, y_val))

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, lc_train, 'o-', color="r", label="Training score")
plt.plot(train_sizes, lc_val, 'o-', color="g", label="Validation score")
plt.title(f"Learning Curve: {FEATURE_NAME} + {CLASSIFIER_NAME}")
plt.xlabel("Training set size fraction")
plt.ylabel("Accuracy Score")
plt.grid(True)
plt.legend()
plt.show()